In [0]:
from pyspark.sql import functions as F

file_path = "/Volumes/workspace/default/classexercise2_netflix_pyspark/netflix_titles.csv"

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .option("multiLine", True)
      .option("escape", "\"")
      .csv(file_path))

df.createOrReplaceTempView("netflix")

df.printSchema()
df.show(5, truncate=False)

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)

+-------+-------+---------------------+---------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------+------------+------+---------+-------------------------------------------------------------+----------------------------------

In [0]:
# DataFrame API
df.groupBy("type").count().show()

# SQL equivalent
# spark.sql("SELECT type, COUNT(*) AS count FROM netflix GROUP BY type").show()

+-------+-----+
|   type|count|
+-------+-----+
|TV Show| 2676|
|  Movie| 6131|
+-------+-----+



In [0]:
# 2. Count of null/missing director values
null_directors = df.filter(F.col("director").isNull() | (F.trim(F.col("director")) == "")).count()
print(f"Number of null/missing director values: {null_directors}")

Number of null/missing director values: 2634


In [0]:
# 3. Count of titles per rating, sorted descending
df.groupBy("rating") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .show(20, truncate=False)

+--------+-----+
|rating  |count|
+--------+-----+
|TV-MA   |3207 |
|TV-14   |2160 |
|TV-PG   |863  |
|R       |799  |
|PG-13   |490  |
|TV-Y7   |334  |
|TV-Y    |307  |
|PG      |287  |
|TV-G    |220  |
|NR      |80   |
|G       |41   |
|TV-Y7-FV|6    |
|NULL    |4    |
|NC-17   |3    |
|UR      |3    |
|84 min  |1    |
|66 min  |1    |
|74 min  |1    |
+--------+-----+



In [0]:
# 4. Top 10 listed_in genres by title count
df.groupBy("listed_in") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .limit(10) \
  .show(truncate=False)

+------------------------------------------------+-----+
|listed_in                                       |count|
+------------------------------------------------+-----+
|Dramas, International Movies                    |362  |
|Documentaries                                   |359  |
|Stand-Up Comedy                                 |334  |
|Comedies, Dramas, International Movies          |274  |
|Dramas, Independent Movies, International Movies|252  |
|Kids' TV                                        |220  |
|Children & Family Movies                        |215  |
|Children & Family Movies, Comedies              |201  |
|Documentaries, International Movies             |186  |
|Dramas, International Movies, Romantic Movies   |180  |
+------------------------------------------------+-----+



In [0]:
# 5. Top 15 individual genres by title count
df.withColumn("genre", F.explode(F.split(F.col("listed_in"), ",\\s*"))) \
  .groupBy("genre") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .limit(15) \
  .show(truncate=False)

+------------------------+-----+
|genre                   |count|
+------------------------+-----+
|International Movies    |2752 |
|Dramas                  |2427 |
|Comedies                |1674 |
|International TV Shows  |1351 |
|Documentaries           |869  |
|Action & Adventure      |859  |
|TV Dramas               |763  |
|Independent Movies      |756  |
|Children & Family Movies|641  |
|Romantic Movies         |616  |
|TV Comedies             |581  |
|Thrillers               |577  |
|Crime TV Shows          |470  |
|Kids' TV                |451  |
|Docuseries              |395  |
+------------------------+-----+



In [0]:
# 6. Type of available content (with percentage share)
total_count = df.count()

df.groupBy("type") \
  .count() \
  .withColumn("percentage", F.round((F.col("count") / total_count) * 100, 2)) \
  .show()

+-------+-----+----------+
|   type|count|percentage|
+-------+-----+----------+
|TV Show| 2676|     30.38|
|  Movie| 6131|     69.62|
+-------+-----+----------+



In [0]:
# 7. Quantity of newly added contents by month and year
df_dates = df.withColumn("parsed_date", F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy"))

df_dates.filter(F.col("parsed_date").isNotNull()) \
  .withColumn("year_added", F.year("parsed_date")) \
  .withColumn("month_added", F.month("parsed_date")) \
  .withColumn("month_name", F.date_format("parsed_date", "MMMM")) \
  .groupBy("year_added", "month_added", "month_name") \
  .count() \
  .orderBy(F.col("year_added").desc(), F.col("month_added").desc()) \
  .show(20)

+----------+-----------+----------+-----+
|year_added|month_added|month_name|count|
+----------+-----------+----------+-----+
|      2021|          9| September|  183|
|      2021|          8|    August|  178|
|      2021|          7|      July|  257|
|      2021|          6|      June|  207|
|      2021|          5|       May|  132|
|      2021|          4|     April|  188|
|      2021|          3|     March|  112|
|      2021|          2|  February|  109|
|      2021|          1|   January|  132|
|      2020|         12|  December|  169|
|      2020|         11|  November|  154|
|      2020|         10|   October|  167|
|      2020|          9| September|  168|
|      2020|          8|    August|  129|
|      2020|          7|      July|  146|
|      2020|          6|      June|  156|
|      2020|          5|       May|  157|
|      2020|          4|     April|  177|
|      2020|          3|     March|  137|
|      2020|          2|  February|  114|
+----------+-----------+----------

In [0]:
# 8. Average duration of TV Shows in seasons
df.filter(F.col("type") == "TV Show") \
  .withColumn("seasons", F.regexp_extract(F.col("duration"), r"(\d+)", 1).cast("int")) \
  .select(F.round(F.avg("seasons"), 2).alias("avg_seasons")) \
  .show()

+-----------+
|avg_seasons|
+-----------+
|       1.76|
+-----------+



In [0]:
# 9. It's Christmas! What movies/TV shows are you binge watching this weekend?
df.filter(
    F.lower(F.col("title")).contains("christmas") |
    F.lower(F.col("description")).contains("christmas")
).select("type", "title", "release_year", "duration", "description") \
 .show(20, truncate=False)

+-------+--------------------------------------+------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|type   |title                                 |release_year|duration |description                                                                                                                                              |
+-------+--------------------------------------+------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|Movie  |Love Is a Story                       |2015        |105 min  |Two young artists, a violin player and an actress, fight to achieve their dreams and keep their relationship together during Christmastime.              |
|TV Show|Home for Christmas                    |2020        |2 Seasons|Tired of the constant com

In [0]:
# 10. Ryan Reynolds movie compilation
df.filter(
    (F.col("type") == "Movie") &
    F.col("cast").contains("Ryan Reynolds")
).select("title", "release_year", "duration", "listed_in") \
 .orderBy(F.col("release_year").desc()) \
 .show(truncate=False)

+-------------------------------------+------------+--------+------------------------------------+
|title                                |release_year|duration|listed_in                           |
+-------------------------------------+------------+--------+------------------------------------+
|6 Underground                        |2019        |129 min |Action & Adventure, Dramas          |
|Betty White: First Lady of Television|2018        |56 min  |Documentaries                       |
|Selfless                             |2015        |117 min |Sci-Fi & Fantasy, Thrillers         |
|Mississippi Grind                    |2015        |109 min |Dramas, Independent Movies          |
|The Captive                          |2014        |112 min |Dramas, Thrillers                   |
|The Croods                           |2013        |99 min  |Children & Family Movies, Comedies  |
|Turbo                                |2013        |96 min  |Children & Family Movies, Comedies  |
|Safe Hous

In [0]:
# 11. Movies or shows released during COVID season (2020 - 2021)
df.filter(F.col("release_year").isin(2020, 2021)) \
  .groupBy("release_year", "type") \
  .count() \
  .orderBy("release_year", "type") \
  .show()

total_covid = df.filter(F.col("release_year").isin(2020, 2021)).count()
print(f"Total titles released during COVID season (2020-2021): {total_covid}")

+------------+-------+-----+
|release_year|   type|count|
+------------+-------+-----+
|        2020|  Movie|  517|
|        2020|TV Show|  436|
|        2021|  Movie|  277|
|        2021|TV Show|  315|
+------------+-------+-----+

Total titles released during COVID season (2020-2021): 1545


In [0]:
# 12. Movies under 2 hours (<= 120 minutes) for a short flight
short_flight_movies = (
    df.filter(F.col("type") == "Movie")
      .withColumn("minutes", F.regexp_extract(F.col("duration"), r"(\d+)", 1).cast("int"))
      .filter(F.col("minutes") <= 120)
      .select("title", "type", "duration", "minutes", "listed_in")
      .orderBy(F.col("minutes").desc())
)

print(f"Total movies under 2 hours (<= 120 min): {short_flight_movies.count()}")
short_flight_movies.show(20, truncate=False)

Total movies under 2 hours (<= 120 min): 4986
+----------------------------------------+-----+--------+-------+--------------------------------------------------------+
|title                                   |type |duration|minutes|listed_in                                               |
+----------------------------------------+-----+--------+-------+--------------------------------------------------------+
|Adú                                     |Movie|120 min |120    |Dramas, International Movies                            |
|The Hows of Us                          |Movie|120 min |120    |Dramas, International Movies, Romantic Movies           |
|Gatao - The Last Stray                  |Movie|120 min |120    |Action & Adventure, Dramas, International Movies        |
|Do the Right Thing                      |Movie|120 min |120    |Classic Movies, Comedies, Dramas                        |
|The Yin Yang Master                     |Movie|120 min |120    |Action & Adventure, Internat